### Load dataset

In [ ]:
import os

import torch
import plotly.express as px
from torch import nn, Tensor
from torchvision.transforms import v2

from src import configs as cfg
from src import models, plotting
from src import dataset, metrics, plotting

In [ ]:
def load_chkpt(chkpt_pth: str) -> torch.nn.Module:
    chkpt = torch.load(chkpt_pth, weights_only=False)
    model_cfg = cfg.ModelConfig(**chkpt["model_cfg"])
    model = models.mk_model_from_cfg(model_cfg)
    model.load_state_dict(chkpt["model"])
    return model

In [ ]:
train_cfg = cfg.TrainingConfig()
loaders = dataset.mk_segmentation_data_loaders(train_cfg)
x, y_true = next(iter(loaders["train"]))
batch = dataset.preprocess_batch({"x": x, "y_true": y_true})

CHKPT_DIRECTORY = "checkpoints/unet/xxx-xxx-xxx"
chkpt_filenames = os.listdir(CHKPT_DIRECTORY)
segs_buffer = torch.empty(
    len(chkpt_filenames), train_cfg.batch_size, 256, 256,
    dtype=torch.uint8,
    device=cfg.DEVICE,
)

for chkpt_idx, chkpt_filename in enumerate(chkpt_filenames):
    chkpt_pth = os.path.join(CHKPT_DIRECTORY, chkpt_filename)
    model = load_chkpt(chkpt_pth)
    segs_buffer[chkpt_idx] = model(batch)["y_pred"].argmax(dim=1)

